# Resizer FPGA Benchmark — Image Resizer
Resize 4K (3840x2160) → 1080p (1920x1080) on **FPGA fabric (PL)**.

**Run from Jupyter** (`http://<board-ip>:9090/lab`) **as root kernel**.

Uses `resizer.bit` from pynq-resizer — a custom FPGA resize IP + AXI DMA.
This is **NOT the DPU** — it's a different FPGA design for image processing.

> ⚠️ Loading this overlay replaces any previously loaded DPU bitstream.
> After running, reload DPU if needed: `sudo bash /home/ubuntu/setup_dpu.sh`

> ⚠️ DMA overhead: direct Python DMA calls add ~10ms vs notebook timeit.
> Published timing uses the official pynq-resizer notebook results.

Expected: **~66.8ms per frame (notebook), ~14.97 FPS, ~3.51W, ~4.27 FPS/Watt**

In [ ]:
import os, time, numpy as np

POWER_SENSOR = "/sys/class/hwmon/hwmon2/power1_input"
OVERLAY_PATH = "/root/jupyter_notebooks/pynq-resizer/resizer.bit"
N_WARMUP = 3
N_RUNS   = 7
IN_W, IN_H   = 3840, 2160
OUT_W, OUT_H = 1920, 1080

def read_power_mw():
    with open(POWER_SENSOR) as f:
        return int(f.read().strip()) / 1000

print(f"Overlay: {OVERLAY_PATH}")
print(f"Exists: {os.path.exists(OVERLAY_PATH)}")
print(f"Current power: {read_power_mw()/1000:.2f} W (idle)")

## Step 1 — Load FPGA Overlay
Programs the FPGA fabric with the resize IP design (~10-20 seconds).

In [ ]:
from pynq import Overlay, allocate
ol = Overlay(OVERLAY_PATH)
resize_ip = ol.resize_accel_0
dma = ol.axi_dma_0
print(f"Overlay loaded! IPs: {list(ol.ip_dict.keys())}")

## Step 2 — Allocate DMA Buffers
Contiguous physical memory required for FPGA DMA transfers.

In [ ]:
in_buf  = allocate(shape=(IN_H, IN_W, 3), dtype=np.uint8)
out_buf = allocate(shape=(OUT_H, OUT_W, 3), dtype=np.uint8)
in_buf[:] = np.random.randint(0, 255, (IN_H, IN_W, 3), dtype=np.uint8)

# Configure resize IP
resize_ip.write(0x10, IN_W)
resize_ip.write(0x18, IN_H)
resize_ip.write(0x20, OUT_W)
resize_ip.write(0x28, OUT_H)
print(f"Input buffer:  {in_buf.shape}")
print(f"Output buffer: {out_buf.shape}")

def resize_frame():
    resize_ip.write(0x00, 1)
    dma.sendchannel.transfer(in_buf)
    dma.recvchannel.transfer(out_buf)
    dma.sendchannel.wait()
    dma.recvchannel.wait()

## Step 3 — Warmup

In [ ]:
for _ in range(N_WARMUP):
    resize_frame()
print("Warmup done!")

## Step 4 — Benchmark with Power

In [ ]:
times_ms = []
power_samples = []
for i in range(N_RUNS):
    p_before = read_power_mw()
    t0 = time.time()
    resize_frame()
    t1 = time.time()
    p_after = read_power_mw()
    times_ms.append((t1 - t0) * 1000)
    power_samples.append((p_before + p_after) / 2)
    print(f"  Run {i+1}: {times_ms[-1]:.1f} ms, {power_samples[-1]/1000:.2f} W")

## Step 5 — Results

In [ ]:
avg_ms = sum(times_ms) / len(times_ms)
avg_power_w = sum(power_samples) / len(power_samples) / 1000
fps = 1000 / avg_ms
fps_per_watt = fps / avg_power_w

print("=" * 40)
print("FPGA RESIZER RESULTS")
print("=" * 40)
print(f"Time/frame:  {avg_ms:.1f} ms  (incl. DMA overhead)")
print(f"FPS:         {fps:.2f}")
print(f"Power:       {avg_power_w:.2f} W")
print(f"FPS/Watt:    {fps_per_watt:.2f}")
print("=" * 40)
print(f"Notebook ref: ~66.8ms, ~14.97 FPS, ~3.51W, ~4.27 FPS/Watt")

del in_buf, out_buf